In [ ]:
from pathlib import Path
import pandas as pd

carpeta_actual = Path.cwd()

if carpeta_actual.name == "notebooks":
    carpeta_proyecto = carpeta_actual.parent
else:
    carpeta_proyecto = carpeta_actual

carpeta_datos = carpeta_proyecto / "datos"
ruta_train = carpeta_datos / "solicitudes_train.csv"

solicitudes = pd.read_csv(ruta_train)

print("Archivo cargado desde:", ruta_train)
solicitudes.head()

,ID_Solicitud,ID_Usuario,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Fecha_Solicitud,Estado_Final
0,1,12698,7900,0.2333,4461,Consumo,2025-04,1
1,2,20744,17400,0.2507,3889,Vivienda,2025-12,1
2,3,6830,8600,0.2368,4035,Vehiculo,2025-06,0
3,4,5722,13300,0.2826,4018,Vehiculo,2025-07,1
4,5,24719,6600,0.2314,3357,Otros,2025-01,1


## Tamaño y tipos de datos

In [ ]:
print("Filas:", solicitudes.shape[0])
print("Columnas:", solicitudes.shape[1])
solicitudes.dtypes

Filas: 32416
Columnas: 8


,0
ID_Solicitud,int64
ID_Usuario,int64
Monto_Solicitado,int64
Tasa_Interes,float64
Ingreso_Mensual,int64
Motivo_Prestamo,object
Fecha_Solicitud,object
Estado_Final,int64


## Valores faltantes y duplicados

In [ ]:
faltantes = pd.DataFrame({
    "cantidad": solicitudes.isna().sum(),
    "porcentaje": solicitudes.isna().mean() * 100
})
faltantes

,cantidad,porcentaje
ID_Solicitud,0,0.0
ID_Usuario,0,0.0
Monto_Solicitado,0,0.0
Tasa_Interes,0,0.0
Ingreso_Mensual,0,0.0
Motivo_Prestamo,0,0.0
Fecha_Solicitud,0,0.0
Estado_Final,0,0.0


In [ ]:
print("Filas duplicadas:", solicitudes.duplicated().sum())
print("ID_Solicitud duplicados:", solicitudes["ID_Solicitud"].duplicated().sum())
print("Usuarios distintos:", solicitudes["ID_Usuario"].nunique())

Filas duplicadas: 0
ID_Solicitud duplicados: 0
Usuarios distintos: 18548


## Estado final

In [ ]:
conteo_estado = solicitudes["Estado_Final"].value_counts().sort_index()

distribucion_estado = pd.DataFrame({
    "cantidad": conteo_estado,
    "porcentaje": conteo_estado / len(solicitudes) * 100
})
distribucion_estado

,cantidad,porcentaje
Estado_Final,,
0,5171,15.951999
1,27245,84.048001


## Variables numéricas

In [ ]:
columnas_numericas = [
    "Monto_Solicitado",
    "Tasa_Interes",
    "Ingreso_Mensual",
]

solicitudes[columnas_numericas].describe().T

,count,mean,std,min,25%,50%,75%,max
Monto_Solicitado,32416.0,14290.097483,12995.295179,500.0000,5700.0000,10200.0000,18100.0000,70000.0000
Tasa_Interes,32416.0,0.259580,0.032333,0.1476,0.2373,0.2583,0.2806,0.3799
Ingreso_Mensual,32416.0,4214.475938,2498.904569,353.0000,2481.0000,3621.0000,5276.0000,22000.0000


In [ ]:
controles = pd.Series({
    "Monto menor o igual a cero": (solicitudes["Monto_Solicitado"] <= 0).sum(),
    "Tasa negativa": (solicitudes["Tasa_Interes"] < 0).sum(),
    "Tasa mayor a 1": (solicitudes["Tasa_Interes"] > 1).sum(),
    "Ingreso menor o igual a cero": (solicitudes["Ingreso_Mensual"] <= 0).sum(),
    "Estado distinto de 0 o 1": (~solicitudes["Estado_Final"].isin([0, 1])).sum(),
})
controles

,0
Monto menor o igual a cero,0
Tasa negativa,0
Tasa mayor a 1,0
Ingreso menor o igual a cero,0
Estado distinto de 0 o 1,0


## Motivo del préstamo

In [ ]:
solicitudes["Motivo_Prestamo"].value_counts(dropna=False)

,count
Motivo_Prestamo,
Consumo,9049
Vivienda,5761
Vehiculo,4256
Otros,3973
Educacion,3876
Salud,2884
Negocio,2617


## Fechas de solicitud

In [ ]:
fechas = pd.to_datetime(solicitudes["Fecha_Solicitud"], errors="coerce")

print("Fechas inválidas:", fechas.isna().sum())
print("Fecha mínima:", fechas.min())
print("Fecha máxima:", fechas.max())

Fechas inválidas: 0
Fecha mínima: 2025-01-01 00:00:00
Fecha máxima: 2025-12-01 00:00:00
